# Đánh giá chất lượng mô hình RAG (Ragas Evaluation)

Notebook này thực hiện quá trình chấm điểm tự động hệ thống Hỏi-Đáp (RAG) bằng phương pháp **LLM-as-a-Judge**. Chúng ta sẽ đánh giá trên 2 tập dữ liệu:
1. **Easy Set**: Các câu hỏi ngắn gọn, trực diện, từ khóa rõ ràng.
2. **Hard Set**: Các câu hỏi dài, mang tính suy luận, lắt léo và đòi hỏi tổng hợp nhiều luồng thông tin.

**Các tiêu chí chấm điểm (Metrics):**
- `Context Recall`: Tài liệu truy xuất có chứa đáp án không?
- `Context Precision`: Tài liệu chứa đáp án có được xếp hạng Top 1 không?
- `Faithfulness`: Câu trả lời của AI có trung thực với tài liệu không (hay bị ảo giác - hallucination)?


## Import các thư viện & Cấu hình đường dẫn 

In [4]:
import sys 
import json 
import nest_asyncio 
import pandas as pd 
from pathlib import Path 

nest_asyncio.apply()
sys.path.append(str(Path.cwd().parent))
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from backend.rag.retriever import retrieve_context
from backend.rag.generator import generate_answer
from backend.core.config import EMBEDDING_MODEL
print(" Đã import các thư viện thành công!")


 Đã import các thư viện thành công!
Đường dẫn hiện tại: None


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8196\1779684474.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision, context_recall
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8196\1779684474.py:12: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_precision, context_recall
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8196\1779684474.py:12: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import con

## Load dữ liệu từ JSON 

In [23]:
DATA_DIR = Path("../data/eval/datasets") 
# # Lấy đường dẫn và in ra màn hình
# my_path = str(Path.cwd().parent)
# print("Đường dẫn hiện tại là:", my_path)

with open(DATA_DIR / "easy_questions.json", "r", encoding="utf-8") as f: 
    easy_questions = json.load(f)

with open(DATA_DIR / "hard_questions.json", "r", encoding="utf-8") as f: 
    hard_questions = json.load(f)


print(f"Đọc dữ liệu từ JSON thành công !!")
df_easy = pd.DataFrame(easy_questions)
display(df_easy)


Đọc dữ liệu từ JSON thành công !!


,question,ground_truth
0,REIS là viết tắt của cụm từ gì?,REIS là viết tắt của Real-time Environmental I...
1,REIS tập trung vào lĩnh vực nào?,REIS là hệ thống giám sát và phân tích dữ liệu...
2,Hệ thống thu thập dữ liệu với tần suất bao nhiêu?,Hệ thống thu thập dữ liệu mỗi 15 phút.
3,REIS giám sát dữ liệu trên phạm vi bao nhiêu t...,REIS giám sát dữ liệu trên toàn bộ 63 tỉnh thà...


## Chuẩn bị và xử lí Dataset cho bước đánh giá 

In [24]:
def prepare_ragas_dataset(question_list): 
    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for item in question_list: 
        q = item["question"]
        print(f"Đang xử lý: {q}")
        retrieved_chunks = retrieve_context(q, k=3, use_hyde=True, use_reranker=True)
        ans = generate_answer(q, retrieved_chunks)
        ctx_texts = [chunk["text"] for chunk in retrieved_chunks]
        
        questions.append(q)
        answers.append(ans)
        contexts.append(ctx_texts)
        ground_truths.append(item["ground_truth"])
        
    data = {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths,
    }
    return Dataset.from_dict(data)

# Chạy với bộ easy 
easy_dataset = prepare_ragas_dataset(easy_questions)
print("Hoàn tất chuẩn bị Dataset Easy!")

Đang xử lý: REIS là viết tắt của cụm từ gì?
Đang sinh câu trả lời giả định (HyDE)...
Sử dụng HyDE Document để search: REIS là viết tắt của cụm từ tiếng Anh "Real Estate Investment System", thường được sử dụng để chỉ cá...
Đang chạy Cross-Encoder Reranker để chấm điểm và xếp hạng lại...
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: REIS tập trung vào lĩnh vực nào?
Đang sinh câu trả lời giả định (HyDE)...
Sử dụng HyDE Document để search: REIS (Research in Engineering and Information Sciences) tập trung chủ yếu vào việc công bố các công ...
Đang chạy Cross-Encoder Reranker để chấm điểm và xếp hạng lại...
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: Hệ thống thu thập dữ liệu với tần suất bao nhiêu?
Đang sinh câu trả lời giả định (HyDE)...
Sử dụng HyDE Document để search: Tần suất thu thập dữ liệu trong hệ thống E-Learning phụ thuộc vào cấu hình của từng mô-đun và mục ti...
Đang chạy Cross-Encoder Reranker để chấm điểm và xếp hạng lại...


KeyboardInterrupt: 

## Chấm điểm RAGAS (trước khi dùng HyDE + Cross Encoder)

In [25]:
def prepare_ragas_dataset_no_advanced(question_list): 
    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for item in question_list: 
        q = item["question"]
        print(f"Đang xử lý: {q}")
        retrieved_chunks = retrieve_context(q, k=3, use_hyde=False, use_reranker=False)
        ans = generate_answer(q, retrieved_chunks)
        ctx_texts = [chunk["text"] for chunk in retrieved_chunks]
        
        questions.append(q)
        answers.append(ans)
        contexts.append(ctx_texts)
        ground_truths.append(item["ground_truth"])
        
    data = {
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths,
    }
    return Dataset.from_dict(data)

# Chạy với bộ easy 
easy_dataset_no_advanced = prepare_ragas_dataset_no_advanced(easy_questions)
print("Hoàn tất chuẩn bị Dataset Easy!")

Đang xử lý: REIS là viết tắt của cụm từ gì?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: REIS tập trung vào lĩnh vực nào?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: Hệ thống thu thập dữ liệu với tần suất bao nhiêu?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Đang xử lý: REIS giám sát dữ liệu trên phạm vi bao nhiêu tỉnh thành?
Đang gửi câu hỏi kèm ngữ cảnh tài liệu cho Gemini...
Hoàn tất chuẩn bị Dataset Easy!


In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall
from ragas.run_config import RunConfig # <--- BẮT BUỘC IMPORT CÁI NÀY

env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

judge_llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.0 
)

judge_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print("Bắt đầu quá trình chấm điểm (Đã bật chế độ chạy chậm chống chặn API)....")

# Cấu hình "Kìm cương" hệ thống: Chỉ cho chạy 1 câu/lần
slow_config = RunConfig(
    max_workers=1, 
    max_wait=120,
)

result_no_advanced = evaluate(
    easy_dataset_no_advanced, 
    metrics = [
        faithfulness, 
        context_precision, 
        context_recall
    ], 
    llm = judge_llm, 
    embeddings = judge_embeddings,
    run_config = slow_config,      # <--- CHÈN VÀO ĐÂY
    raise_exceptions = False       # <--- CHÈN VÀO ĐÂY ĐỂ BẮT LỖI
)

print(f"Điểm số bộ easy (No Advanced)")
print(result_no_advanced)

# Tự động tạo thư mục ở root project (lùi ra 1 bước bằng ../)
os.makedirs("../reports/ragas", exist_ok=True)

# LƯU Ý: Phải gọi to_pandas() từ biến result_no_advanced
df_no_advanced = result_no_advanced.to_pandas()
df_no_advanced.to_csv("../reports/ragas/ragas_baseline_easy_no_advanced.csv", index=False, encoding='utf-8-sig')

print("✅ Đã lưu kết quả chi tiết vào ../reports/ragas/ragas_baseline_easy_no_advanced.csv")


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8196\3938709742.py:7: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, context_precision, context_recall
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8196\3938709742.py:7: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulness, context_precision, context_recall
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8196\3938709742.py:7: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import contex

Bắt đầu quá trình chấm điểm (Đã bật chế độ chạy chậm chống chặn API)....


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

Exception raised in Job[4]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqbysgh1fmdrh2k1w1md24ge` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98954, Requested 1267. Please try again in 3m10.944s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[5]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqbysgh1fmdrh2k1w1md24ge` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 98761, Requested 1614. Please try again in 5m24s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[6]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit rea

KeyboardInterrupt: 

Exception raised in Job[7]: TimeoutError()
Exception raised in Job[8]: AssertionError(set LLM before use)
Exception raised in Job[9]: AssertionError(LLM is not set)
Exception raised in Job[10]: AssertionError(LLM is not set)
Exception raised in Job[11]: AssertionError(set LLM before use)


## Chấm điểm RAGAS (sau khi dùng HyDE + Cross Encoder)

In [19]:
import os
from dotenv import load_dotenv
env_path = Path("../.env")
load_dotenv(dotenv_path=env_path)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
judge_llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.0 
)

judge_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print("Bắt đầu quá trình chấm điểm....")

result = evaluate(
    easy_dataset, 
    metrics = [
        faithfulness, 
        context_precision, 
        context_recall
    ], 
    llm = judge_llm, 
    embeddings = judge_embeddings
)
print(f"Điểm số bộ easy")
print(result)

# Tự động tạo thư mục ở root project (lùi ra 1 bước bằng ../)
os.makedirs("../reports/ragas", exist_ok=True)

# Lưu ra file ở root project
df_easy.to_csv("../reports/ragas/ragas_baseline_easy.csv", index=False, encoding='utf-8-sig')
print("✅ Đã lưu kết quả chi tiết vào ../reports/ragas/ragas_baseline_easy.csv")


Bắt đầu quá trình chấm điểm....


Evaluating:   0%|          | 0/45 [00:00<?, ?it/s]

Exception raised in Job[10]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqbysgh1fmdrh2k1w1md24ge` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99326, Requested 1413. Please try again in 10m38.496s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[13]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kqbysgh1fmdrh2k1w1md24ge` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99318, Requested 1607. Please try again in 13m19.199999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}})
Exception raised in Job[19]: RateLimitError(Error code: 429 - {'error': {'message': 

Điểm số bộ easy
{'faithfulness': 0.6667, 'context_precision': nan, 'context_recall': 1.0000}
✅ Đã lưu kết quả chi tiết vào ../reports/ragas/ragas_baseline_easy.csv
